# Process MVR and VC Cascade

**Learning outcome:** Apply process mvr and vc cascade through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Advanced  
**Execution profile:** `slow-hpr`  
**Expected runtime:** 5 to 30 minutes  
**Optional extras:** hpr

The lifecycle is explicit: prepare the study, run the named method, then inspect cached results. Observation cells do not launch analysis.

## Study question and data

**Study question:** How does direct process-vapour recompression change streams and the resulting heat-integration target?

The sample data is packaged with OpenPinch, so the notebook runs without path setup. Read the named inputs and assumptions before substituting plant data.

## Step 1: Add and target a process MVR component

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
from OpenPinch import PinchProblem

problem = PinchProblem("process_mvr.json", project_name="Site")
mvr = problem.components.add_process_mvr(
    "Evaporator vapour",
    liquid_injection=False,
    compressor_efficiency=0.72,
    motor_efficiency=0.96,
)
target = problem.target.direct_heat_integration(
    zone="Evaporation Train"
)
mvr_summary = problem.summary_frame()

## Step 2: Inspect lifecycle and compare cascade targeting

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
component_inventory = problem.components.inventory
process_components = problem.process_components
component_is_active = mvr.active
component_type = mvr.component_type
original_streams = mvr.original_streams
replacement_streams = mvr.replacement_streams
stage_results = mvr.stage_results_by_period
affected_zones = mvr.affected_zone_paths
compressor_work = mvr.work_for_zone(problem.master_zone)
try:
    cascade = problem.target.mvr_heat_pump(
        load_fraction=0.25, maximum_restarts=1
    )
except (ValueError, RuntimeError, NotImplementedError) as error:
    cascade = {"status": "no feasible solution", "reason": str(error)}
mvr.deactivate()
mvr.activate()
component_inventory

## Review the result

Review the targeted study, component inventory, stage results, and cascade outcome together to check that compression work and replacement streams serve the intended process zone.

In [ ]:
from IPython.display import display

display(mvr_summary)
display(component_inventory)
display(stage_results)
display(cascade)

## Interpret the result

Audit original and replacement streams, compressor work, affected zones, and active state before comparing targets.

## Adapt this template

Select vapour streams by stable names, expose efficiency assumptions, and keep one component id per physical proposal.

Keep the workflow explicit: prepare input, call one named engineering method, inspect cached results, then export.